# Train YOLO26n on the rocket-parts dataset

Before running: **Runtime → Change runtime type → T4 GPU**.

Add your Roboflow private key as a Colab secret: click the **key icon** in the left sidebar → *Add new secret* → name `ROBOFLOW_API_KEY`, paste the key, and turn on *Notebook access*. (Never paste the key into a cell.)

Then **Runtime → Run all**. Training takes roughly 1–2 hours on a free T4. Everything important is saved to your Google Drive in `drone_yolo/`, so a disconnect doesn't lose your work.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip install -q ultralytics roboflow

## 1. Save results to Google Drive (survives Colab disconnects)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
SAVE_DIR = '/content/drive/MyDrive/drone_yolo'
import os; os.makedirs(SAVE_DIR, exist_ok=True)

## 2. Download dataset version 1 from Roboflow (YOLO format)

In [ ]:
from google.colab import userdata
from roboflow import Roboflow
rf = Roboflow(api_key=userdata.get('ROBOFLOW_API_KEY'))
dataset = rf.workspace('anant-grover').project('real_drone_data').version(1).download('yolov8')
DATA_YAML = f'{dataset.location}/data.yaml'
print(open(DATA_YAML).read())

## 3. Train

- `yolo26n.pt`: the nano model, small and fast enough for the Pi's AI HAT.
- `patience=20`: stops early if validation stops improving.
- Roboflow already added flips/brightness/blur; Ultralytics adds its own mosaic/scale augmentations on top.

In [ ]:
from ultralytics import YOLO
model = YOLO('yolo26n.pt')
results = model.train(
    data=DATA_YAML,
    epochs=80,
    imgsz=640,
    batch=32,
    patience=20,
    project=SAVE_DIR,
    name='run1',
    exist_ok=True,
)

## 4. Honest score on the held-out test photos

The test set is whole photo sessions (IMG_20xx–21xx and DJI_0230–0255) the model never saw. Look at **mAP50** overall and per class, especially *nosecone* (smallest part) and *body tube* (what the claw grabs).

In [ ]:
best = YOLO(f'{SAVE_DIR}/run1/weights/best.pt')
metrics = best.val(data=DATA_YAML, split='test', imgsz=640)
print('test mAP50   :', round(metrics.box.map50, 3))
print('test mAP50-95:', round(metrics.box.map, 3))
for i, name in best.names.items():
    p, r, ap50, ap = metrics.box.class_result(i)
    print(f'  {name:10s} precision={p:.2f}  recall={r:.2f}  mAP50={ap50:.3f}')

## 5. Look at predictions on a few test images

In [ ]:
import glob
from IPython.display import Image, display
test_imgs = sorted(glob.glob(f'{dataset.location}/test/images/*'))[:8]
preds = best.predict(test_imgs, imgsz=640, conf=0.25, save=True, project=SAVE_DIR, name='test_preds', exist_ok=True)
for p in sorted(glob.glob(f'{SAVE_DIR}/test_preds/*'))[:8]:
    display(Image(filename=p, width=500))

## 6. Export for the Raspberry Pi AI HAT

The AI HAT (Hailo chip) needs a `.hef` file. That's a second step done with Hailo's Dataflow Compiler on an x86 Linux machine, starting from this ONNX file. We'll do that step together next.

In [ ]:
onnx_path = best.export(format='onnx', imgsz=640, opset=11, simplify=True)
import shutil; shutil.copy(onnx_path, f'{SAVE_DIR}/best.onnx')
print('Saved to Google Drive:', f'{SAVE_DIR}/run1/weights/best.pt', 'and', f'{SAVE_DIR}/best.onnx')